# Crime Trend Prediction with a Temporal Convolutional Network (TCN)

This notebook replaces the LSTM in `crime_prediction_week7.ipynb` with a **TCN** — a model that uses 1-D dilated causal convolutions instead of recurrent cells.

**Why TCN for this task?**
- Our input is always a fixed 12-month window → LSTM's unbounded memory is wasted; TCN's *explicit* receptive field fits naturally.
- TCN trains in parallel across all time steps; LSTM must unroll sequentially.
- Residual shortcuts give cleaner gradient flow than LSTM gates.

**Output contract (identical to the LSTM notebook):**
```python
{ 'predicted_count': float, 'trend_direction': 'up'|'down'|'stable', 'confidence_score': float }
```

Every code cell has comments explaining not just *what* the code does but *why* each decision was made.

In [ ]:
# ── Standard imports ─────────────────────────────────────────────────────────
import math
import os
import warnings

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

# PyTorch core — the entire framework lives under `torch`.
# torch.nn   : building blocks for neural networks (layers, loss functions)
# torch.utils.data : Dataset and DataLoader abstractions
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

# ── Reproducibility ───────────────────────────────────────────────────────────
# PyTorch and NumPy both use pseudo-random number generators.
# Setting seeds makes weight initialization and data shuffling identical
# every run, so experiments are comparable.
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# ── Device selection ──────────────────────────────────────────────────────────
# Tensors and model parameters live on a 'device'. GPU (cuda) is much faster
# for training. On a MacBook without an NVIDIA GPU this will show 'cpu' —
# that is fine for this dataset size.
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## Step 1 — Load Data

The PySpark ETL produced `crime_aggregations/csv/data.csv` with one row per
`(community_id, year, month, primary_type)`. We need **total** monthly crimes
per area, so we sum across all crime types before any modelling work.

In [ ]:
DATA_PATH = '/Users/yeelingliang/Downloads/CityData/projectDatabases/crime_aggregations/csv/data.csv'

raw = pd.read_csv(DATA_PATH)
print('Raw shape:', raw.shape)
print('Columns:', list(raw.columns))
print('Years:', sorted(raw['year'].unique()))
print('Community areas:', raw['community_id'].nunique())

# Sum crime_count across all primary_type values for each (area, year, month).
# The TCN predicts *total* monthly crimes — type breakdown is interpretation, not input.
monthly_df = (
    raw
    .groupby(['community_id', 'year', 'month'], as_index=False)['crime_count']
    .sum()
    .rename(columns={'community_id': 'community_area', 'year': 'Year', 'month': 'Month'})
    .sort_values(['community_area', 'Year', 'Month'])
    .reset_index(drop=True)
)

print(f'\nAggregated: {monthly_df.shape}')
print(f'Expected dense: {77 * monthly_df["Year"].nunique() * 12} rows')
monthly_df.head(8)

## Step 2 — Exploratory Data Analysis

Before building any model, look for patterns the TCN should learn:
seasonal rhythms, the 2020 COVID disruption, and the spread of crime counts
across all 77 community areas.

In [ ]:
# Hyde Park is community area 41.
hyde = monthly_df[monthly_df['community_area'] == 41].copy()
hyde['date'] = pd.to_datetime(dict(year=hyde['Year'], month=hyde['Month'], day=1))
hyde_mean = hyde['crime_count'].mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(hyde['date'], hyde['crime_count'], marker='o', linewidth=2, label='Monthly crimes')
ax.axhline(hyde_mean, color='black', linestyle='--', linewidth=1.5,
           label=f'Mean: {hyde_mean:.1f}')
ax.axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2020-09-30'),
           color='orange', alpha=0.2, label='COVID disruption')
ax.set_title('Hyde Park (CA 41) Monthly Crime Counts', fontsize=14)
ax.set_xlabel('Month')
ax.set_ylabel('Crime count')
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: average crime count by community area (rows) and calendar month (columns).
# Summer months (6-8) are visibly brighter for most areas — a seasonal pattern
# the TCN should learn.
heatmap_df = (
    monthly_df
    .groupby(['community_area', 'Month'])['crime_count']
    .mean()
    .reset_index()
    .pivot(index='community_area', columns='Month', values='crime_count')
)

fig, ax = plt.subplots(figsize=(12, 14))
sns.heatmap(heatmap_df, cmap='viridis', ax=ax,
            cbar_kws={'label': 'Avg monthly crimes'})
ax.set_title('Average Monthly Crime by Community Area and Month', fontsize=12)
ax.set_xlabel('Month')
ax.set_ylabel('Community Area')
plt.tight_layout()
plt.show()

## Step 2b — ACF Analysis: Validating the 12-Month Window

Before hardcoding `SEQ_LEN = 12`, we should ask: *how far back does the past actually predict the future?*

**What is ACF (Autocorrelation Function)?**  
ACF measures the correlation between a time series and a lagged copy of itself.
- `ACF(lag=1)` = correlation between month *t* and month *t-1*
- `ACF(lag=12)` = correlation between month *t* and month *t-12* (same month last year)

A high positive value means "the past at that lag is a strong predictor of the present."  
The shaded band is the 95% confidence interval — bars outside it are statistically significant lags.

**Decision rule:**  
Set `SEQ_LEN` to the last lag where the ACF bar clearly exits the confidence band.  
If that lag is ≤ 12, our choice is conservative (safe).  
If that lag is > 12, we are cutting off real signal.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

# ── Choose representative community areas ─────────────────────────────────────
# We run ACF on three CAs that differ in crime volume:
#   CA 41 = Hyde Park      (medium crime, near UChicago)
#   CA 8  = Near North Side (low crime, wealthy lakefront)
#   CA 25 = Austin          (high crime, West Side)
# If all three show significance dropping off at roughly the same lag,
# that lag is a dataset-wide property, not specific to one neighbourhood.
EXAMPLE_CAS = {41: 'Hyde Park', 8: 'Near North Side', 25: 'Austin'}

# We plot 36 lags (3 years) so we can see whether significance persists
# beyond one seasonal cycle (12 months).
N_LAGS = 36

fig, axes = plt.subplots(len(EXAMPLE_CAS), 1, figsize=(14, 4 * len(EXAMPLE_CAS)))

last_significant_lags = {}  # will store the finding per CA for the decision below

for ax, (ca, name) in zip(axes, EXAMPLE_CAS.items()):
    # Extract the raw crime_count series for this CA, sorted by time.
    series = (
        monthly_df[monthly_df['community_area'] == ca]
        .sort_values(['Year', 'Month'])['crime_count']
        .reset_index(drop=True)
    )

    # plot_acf draws vertical bars for each lag and a shaded 95% confidence band.
    # alpha=0.05 → the band width = ±1.96 / sqrt(N), where N = len(series).
    # Any bar that exits the band is statistically significant at p < 0.05.
    plot_acf(
        series,
        lags=N_LAGS,
        alpha=0.05,       # 95% confidence band
        ax=ax,
        zero=False,       # lag-0 autocorrelation is always 1.0; hiding it keeps the y-axis readable
    )
    ax.set_title(f'ACF — CA {ca}: {name}  (n={len(series)} months)', fontsize=12)
    ax.set_xlabel('Lag (months)')
    ax.set_ylabel('Autocorrelation')

    # ── Find the last statistically significant lag ───────────────────────────
    # plot_acf returns a Lines object; the confidence interval values are stored
    # in the third line (index 2) of the axes' lines after the call.
    # A simpler approach: compute it directly from the series using acf().
    from statsmodels.tsa.stattools import acf
    acf_values, confint = acf(series, nlags=N_LAGS, alpha=0.05)

    # confint[lag] = [lower_bound, upper_bound] of the 95% CI for that lag.
    # A lag is significant if the ACF value falls OUTSIDE [lower, upper].
    # We skip lag 0 (always 1.0) by starting at index 1.
    significant_lags = [
        lag for lag in range(1, N_LAGS + 1)
        if not (confint[lag][0] <= acf_values[lag] <= confint[lag][1])
    ]

    last_sig = max(significant_lags) if significant_lags else 0
    last_significant_lags[ca] = last_sig

    # Draw a vertical reference line at lag 12 so we can visually judge
    # whether our chosen SEQ_LEN sits inside or outside the significance zone.
    ax.axvline(12, color='red', linestyle='--', linewidth=1.5,
               label=f'Current SEQ_LEN=12  |  last significant lag={last_sig}')
    ax.legend(loc='upper right')

plt.tight_layout()
plt.show()

# ── Decision summary ──────────────────────────────────────────────────────────
# Print a plain-English verdict for each CA so the reader doesn't have to
# interpret the plot themselves.
print('ACF window-size verdict\n' + '─' * 40)
recommended = []
for ca, name in EXAMPLE_CAS.items():
    last_sig = last_significant_lags[ca]
    verdict = (
        'SEQ_LEN=12 CAPTURES ALL SIGNAL  ✓'
        if last_sig <= 12
        else f'SIGNAL EXTENDS TO {last_sig} MONTHS — consider SEQ_LEN={last_sig}  ⚠'
    )
    print(f'  CA {ca:2d} ({name:20s}): last significant lag = {last_sig:2d}  →  {verdict}')
    recommended.append(last_sig)

# The conservative choice: use the maximum last-significant lag across all CAs
# so that no community area is left with truncated history.
recommended_seq_len = max(recommended)
print(f'\nRecommended SEQ_LEN (max across sampled CAs): {recommended_seq_len}')
print(f'Current    SEQ_LEN: 12')
if recommended_seq_len <= 12:
    print('→ 12-month window is validated. Proceeding.')
else:
    print(f'→ Consider changing SEQ_LEN to {recommended_seq_len} in Step 3 onwards.')

## Step 3 — Feature Engineering

We build 10 features per row. The TCN model sees these features as a sequence
of 12 consecutive months and predicts the total crime count for month 13.

| Feature | What it captures |
|---|---|
| lag_1 … lag_12 | How many crimes happened 1, 2, 3, 6, 12 months ago |
| rolling_3, rolling_6 | Short-term and medium-term trend |
| month_sin, month_cos | Seasonal position (circular encoding) |
| community_area | Which neighborhood (identity feature) |

**Critical rule:** always sort by `(community_area, Year, Month)` before calling
`shift()` or `rolling()`. If the sort is wrong, lag features bleed across areas.

In [ ]:
monthly_df = monthly_df.sort_values(['community_area', 'Year', 'Month']).reset_index(drop=True)

# ── Lag features ──────────────────────────────────────────────────────────────
# shift(N) inside groupby moves values DOWN by N rows within each group.
# lag_1 at month t = crime_count at month t-1 (same community area).
# lag_12 at month t = crime_count at the same calendar month last year.
for lag in [1, 2, 3, 6, 12]:
    monthly_df[f'lag_{lag}'] = (
        monthly_df.groupby('community_area')['crime_count'].shift(lag)
    )

# ── Rolling averages ──────────────────────────────────────────────────────────
# rolling(N).mean() is the average of the last N values.
# min_periods=1 means we keep the computation even at the start of a series
# (rather than producing NaN for the first N-1 rows).
monthly_df['rolling_3'] = (
    monthly_df.groupby('community_area')['crime_count']
    .transform(lambda x: x.rolling(3, min_periods=1).mean())
)
monthly_df['rolling_6'] = (
    monthly_df.groupby('community_area')['crime_count']
    .transform(lambda x: x.rolling(6, min_periods=1).mean())
)

# ── Seasonal encoding ─────────────────────────────────────────────────────────
# Encoding month as a raw integer (1-12) is wrong: December (12) and January (1)
# appear 11 steps apart but are actually consecutive months.
# A sin/cos pair maps each month onto a unit circle where December and January
# are geometrically adjacent. The model learns smoother seasonal patterns.
monthly_df['month_sin'] = np.sin(2 * np.pi * monthly_df['Month'] / 12)
monthly_df['month_cos'] = np.cos(2 * np.pi * monthly_df['Month'] / 12)

# ── Target ────────────────────────────────────────────────────────────────────
# shift(-1) moves values UP: target at row t = crime_count at row t+1.
# This means: given this month's features, predict NEXT month's total crimes.
monthly_df['target'] = (
    monthly_df.groupby('community_area')['crime_count'].shift(-1)
)

# Drop rows where lag_12 is NaN (first 12 months of each area's history)
# and where target is NaN (the final month of each area's series).
LAG_COLS = ['lag_1', 'lag_2', 'lag_3', 'lag_6', 'lag_12']
feature_df = monthly_df.dropna(subset=LAG_COLS + ['target']).reset_index(drop=True)

FEATURE_COLS = [
    'lag_1', 'lag_2', 'lag_3', 'lag_6', 'lag_12',
    'rolling_3', 'rolling_6', 'month_sin', 'month_cos', 'community_area',
]

print('Feature rows:', feature_df.shape)
print('Features:', FEATURE_COLS)
print('Years available:', sorted(feature_df['Year'].unique()))
feature_df.head()

## Step 4 — Naive Baseline

Every ML model needs a sanity check: if the model can't beat a simple rule,
it is not adding value. The baseline here is **lag-12**: predict that next
month will look like the same month last year.

In [ ]:
def rmse_score(actuals, preds):
    """Compatibility wrapper for older and newer scikit-learn."""
    try:
        return mean_squared_error(actuals, preds, squared=False)
    except TypeError:
        return math.sqrt(mean_squared_error(actuals, preds))

test_naive   = feature_df[feature_df['Year'] == 2024].copy()
naive_preds  = test_naive['lag_12'].values
naive_actuals = test_naive['target'].values

naive_mae  = mean_absolute_error(naive_actuals, naive_preds)
naive_rmse = rmse_score(naive_actuals, naive_preds)
print(f'Naive baseline  MAE: {naive_mae:.2f}   RMSE: {naive_rmse:.2f}')

## Step 5 — Train / Val / Test Split + Scaling

We split by time, not randomly. Randomly shuffling time-series data leaks future
information into the past and produces falsely optimistic metrics.

- **Train:** years ≤ 2022  
- **Val:** 2023 (tune hyperparameters here; never touch test)  
- **Test:** 2024 (final score, look once at the end)

`StandardScaler` is fit **only on train rows**. Using val/test statistics in
`fit_transform` would let the model 'see' the future mean and variance.

In [ ]:
train_rows = feature_df[feature_df['Year'] <= 2022].copy()
val_rows   = feature_df[feature_df['Year'] == 2023].copy()
test_rows  = feature_df[feature_df['Year'] == 2024].copy()

# fit_transform: compute mean/std from train, then scale train.
# transform:     apply the SAME mean/std to val and test (no new fitting).
scaler = StandardScaler()
train_rows[FEATURE_COLS] = scaler.fit_transform(train_rows[FEATURE_COLS])
val_rows[FEATURE_COLS]   = scaler.transform(val_rows[FEATURE_COLS])
test_rows[FEATURE_COLS]  = scaler.transform(test_rows[FEATURE_COLS])

# Unscaled copy kept for predict() — we scale on demand there.
feature_df_unscaled = feature_df.copy()

print('Train:', train_rows.shape, '| Years:', sorted(train_rows['Year'].unique()))
print('Val:  ', val_rows.shape,   '| Year: ', sorted(val_rows['Year'].unique()))
print('Test: ', test_rows.shape,  '| Year: ', sorted(test_rows['Year'].unique()))

## Step 6 — Dataset and DataLoader

PyTorch's training loop expects a `DataLoader` which batches data from a `Dataset`.

Our `CrimeDataset` creates **sliding windows**: for each community area with T months
of feature rows, it produces (T − seq_len) training examples. Each example is a
`(seq_len, n_features)` tensor paired with a scalar target.

In [ ]:
SEQ_LEN = 12  # 12-month lookback window

class CrimeDataset(Dataset):
    """
    Produces sliding (seq_len, n_features) windows from a monthly crime DataFrame.

    PyTorch requires any Dataset to implement __len__ and __getitem__.
    DataLoader calls __getitem__ repeatedly and collates results into batches.
    """
    def __init__(self, frame, feature_cols, seq_len=12, target_years=None):
        self.samples = []
        target_years = set(target_years) if target_years is not None else None

        ordered = frame.sort_values(['community_area', 'Year', 'Month']).reset_index(drop=True)

        for ca, ca_frame in ordered.groupby('community_area'):
            ca_frame = ca_frame.reset_index(drop=True)
            if len(ca_frame) < seq_len + 1:
                continue  # Not enough rows for even one complete window

            feats   = ca_frame[feature_cols].values.astype(np.float32)
            targets = ca_frame['target'].values.astype(np.float32)
            years   = ca_frame['Year'].values.astype(int)
            months  = ca_frame['Month'].values.astype(int)

            # Slide the window: each end_idx gives one training example.
            for end_idx in range(seq_len - 1, len(ca_frame)):
                if target_years is not None and years[end_idx] not in target_years:
                    continue
                start_idx = end_idx - seq_len + 1
                self.samples.append({
                    'x':             feats[start_idx:end_idx + 1],  # (seq_len, n_features)
                    'y':             targets[end_idx],
                    'community_area': int(ca),
                    'year':          years[end_idx],
                    'month':         months[end_idx],
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        # torch.tensor wraps a numpy array into a PyTorch tensor.
        # dtype=float32 is required — nn.Linear and Conv1d expect float32.
        # unsqueeze not needed here because DataLoader stacks items into batches
        # automatically: N items of shape (12, 10) → (N, 12, 10).
        return (
            torch.tensor(s['x'], dtype=torch.float32),
            torch.tensor([s['y']], dtype=torch.float32),
        )

    def metadata(self):
        return pd.DataFrame([{k: v for k, v in s.items() if k != 'x'} for s in self.samples])


def make_dataset_with_context(prior_rows, current_rows, feature_cols, seq_len):
    """
    Prepend `seq_len` prior rows per CA so that val/test windows have enough
    lookback history without including out-of-split targets.
    """
    parts = []
    for ca in sorted(current_rows['community_area'].unique()):
        ca_prior   = prior_rows[prior_rows['community_area'] == ca].tail(seq_len)
        ca_current = current_rows[current_rows['community_area'] == ca]
        if len(ca_current):
            parts.append(pd.concat([ca_prior, ca_current], ignore_index=True))
    combined = (
        pd.concat(parts, ignore_index=True)
        .sort_values(['community_area', 'Year', 'Month'])
        .reset_index(drop=True)
    )
    return CrimeDataset(combined, feature_cols, seq_len,
                        target_years=current_rows['Year'].unique())


train_dataset = CrimeDataset(train_rows, FEATURE_COLS, SEQ_LEN)
val_dataset   = make_dataset_with_context(train_rows, val_rows, FEATURE_COLS, SEQ_LEN)
test_dataset  = make_dataset_with_context(
    pd.concat([train_rows, val_rows]), test_rows, FEATURE_COLS, SEQ_LEN
)

print('Train sequences:', len(train_dataset))
print('Val sequences:  ', len(val_dataset))
print('Test sequences: ', len(test_dataset))

## Step 7 — TCN Architecture

A TCN has three building blocks. Each is a separate `nn.Module`.

```
Chomp1d          → strips future-lookahead padding to enforce causality
TemporalBlock    → one residual block: two dilated causal convolutions + skip connection
CrimeTCN         → stacks 4 TemporalBlocks with dilations [1, 2, 4, 8]
```

We build them bottom-up.

In [ ]:
class Chomp1d(nn.Module):
    """
    Removes the last `chomp_size` positions from the time axis of a Conv1d output.

    WHY THIS IS NECESSARY:
    A causal convolution must only see past and present timesteps — never future ones.

    nn.Conv1d with padding=P adds P zeros on the LEFT *and* P zeros on the RIGHT
    of the sequence (symmetric padding). That means output[t] can peek at
    input[t+1], ..., input[t+P]: a data leak.

    Fix: set padding = (kernel_size - 1) * dilation in Conv1d (which PyTorch places
    symmetrically), then slice off the right-side padding with Chomp1d.

    After Chomp1d:
      input length  = L
      Conv1d output = L + 2*P - dilation*(kernel_size-1)
                    = L + 2*(k-1)*d - (k-1)*d = L + (k-1)*d
      After chomp   = L  ← same as input. ✓ Causal. ✓ Length preserved.

    Example (kernel_size=3, dilation=2):
      padding = (3-1)*2 = 4
      output[t] uses input[t-4], input[t-2], input[t]  ← all past or present ✓
    """
    def __init__(self, chomp_size: int):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, channels, seq_len + chomp_size)
        # Slice the last chomp_size steps off the time axis (dim=2).
        # .contiguous() ensures the memory layout is standard after slicing
        # (non-contiguous tensors can crash some subsequent ops).
        return x[:, :, :-self.chomp_size].contiguous()

In [ ]:
class TemporalBlock(nn.Module):
    """
    One residual block of the TCN.

    STRUCTURE:
      Input (batch, in_channels, seq_len)
         |
         +---> Conv1d (causal, dilation=d) -> Chomp -> ReLU -> Dropout
         |              |
         |     Conv1d (causal, dilation=d) -> Chomp -> ReLU -> Dropout
         |              |
         +---> 1x1 Conv if in_ch != out_ch
                |       |
                +- ADD -+
                    |
                  ReLU
      Output (batch, out_channels, seq_len)

    KEY DESIGN DECISIONS:

    1. Two conv layers per block (Bai et al. 2018): doubles the effective
       receptive field per block while keeping architecture modular.

    2. Weight normalization (not BatchNorm):
       - BatchNorm divides by the batch mean/std. With batch_size=1 (as in
         predict() at inference time), this estimate is meaningless.
       - WeightNorm reparameterizes W = g * (V / ||V||): g is a learned
         magnitude scalar, V is the learned direction matrix. No batch stats
         needed → stable at any batch size, including 1.

    3. Residual connection: output = F(x) + x
       - d(loss)/d(x) = dF/dx + 1  (via chain rule, the +1 comes from the skip)
       - The '+1' ensures gradient >= 1 at every block → no vanishing.
       - When in_channels != out_channels, a 1x1 conv projects x to the right
         shape without mixing temporal positions.

    4. Dropout AFTER activation: dropping before ReLU zeros out values that
       were already zero — wasteful. After ReLU, dropout adds genuine stochasticity.
    """
    def __init__(self, in_channels: int, out_channels: int,
                 kernel_size: int, dilation: int, dropout: float = 0.2):
        super().__init__()
        # Padding that makes Conv1d output the same length as input (before Chomp).
        padding = (kernel_size - 1) * dilation

        # nn.utils.weight_norm wraps Conv1d in-place, replacing 'weight' with
        # two learnable tensors: 'weight_g' (magnitude) and 'weight_v' (direction).
        self.conv1    = nn.utils.weight_norm(nn.Conv1d(
            in_channels, out_channels, kernel_size,
            padding=padding, dilation=dilation,
        ))
        self.chomp1   = Chomp1d(padding)
        self.relu1    = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2    = nn.utils.weight_norm(nn.Conv1d(
            out_channels, out_channels, kernel_size,
            padding=padding, dilation=dilation,
        ))
        self.chomp2   = Chomp1d(padding)
        self.relu2    = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        # 1x1 conv projects channels when in != out. kernel_size=1 means each
        # timestep is transformed independently — no temporal mixing on the skip path.
        self.residual = (
            nn.Conv1d(in_channels, out_channels, 1)
            if in_channels != out_channels else None
        )
        self.final_relu = nn.ReLU()
        self._init_weights()

    def _init_weights(self):
        # Small std (0.01) keeps initial activations small → stable early training.
        nn.init.normal_(self.conv1.weight, 0, 0.01)
        nn.init.normal_(self.conv2.weight, 0, 0.01)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, in_channels, seq_len)
        out = self.dropout1(self.relu1(self.chomp1(self.conv1(x))))
        out = self.dropout2(self.relu2(self.chomp2(self.conv2(out))))
        shortcut = self.residual(x) if self.residual is not None else x
        return self.final_relu(out + shortcut)

In [ ]:
class CrimeTCN(nn.Module):
    """
    Full TCN for monthly crime count prediction.

    DATA FLOW:
      Input:  (batch, seq_len=12, n_features=10)  <- DataLoader format
      Transpose: (batch, n_features=10, seq_len=12)  <- Conv1d needs channels-first
      TemporalBlock(dilation=1) -> (batch, 64, 12)
      TemporalBlock(dilation=2) -> (batch, 64, 12)
      TemporalBlock(dilation=4) -> (batch, 64, 12)
      TemporalBlock(dilation=8) -> (batch, 64, 12)
      Last timestep [:, :, -1]  -> (batch, 64)
      Linear(64 -> 1)           -> (batch, 1)   predicted count

    RECEPTIVE FIELD MATH:
    With kernel_size=3 and dilations [1, 2, 4, 8]:
      RF = 1 + (kernel_size - 1) * 2 * sum(dilations)
         = 1 + 2 * 2 * (1 + 2 + 4 + 8)
         = 1 + 4 * 15 = 61 months

    Our input is only 12 months, so the RF > window: every input timestep
    contributes to the output. This is intentional — we want capacity to
    integrate the full history without the model being forced to.

    EXPONENTIAL DILATION (2^i):
    Doubling dilation at each level gives exponentially growing RF at zero
    extra parameter cost.
      4 blocks, kernel=3, no dilation:   RF = 9 months
      4 blocks, kernel=3, dilation x2:  RF = 61 months
    Same parameter count, 6.8x more temporal reach.
    """
    def __init__(
        self,
        n_features: int,
        num_channels: list = None,
        kernel_size: int = 3,
        dropout: float = 0.2,
    ):
        super().__init__()
        if num_channels is None:
            num_channels = [64, 64, 64, 64]

        layers = []
        for i, out_ch in enumerate(num_channels):
            in_ch    = n_features if i == 0 else num_channels[i - 1]
            dilation = 2 ** i  # 1, 2, 4, 8
            layers.append(TemporalBlock(in_ch, out_ch, kernel_size, dilation, dropout))

        # nn.Sequential: forward() applies each layer in list order.
        self.network = nn.Sequential(*layers)
        # Regression head: maps the last-timestep feature vector to one number.
        self.fc = nn.Linear(num_channels[-1], 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Input arrives as (batch, seq_len, features) from DataLoader.
        # Conv1d needs (batch, features, seq_len) — swap axes 1 and 2.
        y = x.transpose(1, 2)    # (batch, features, seq_len)
        y = self.network(y)      # (batch, 64, seq_len)
        y = y[:, :, -1]          # (batch, 64) — only the final timestep
        return self.fc(y)        # (batch, 1)

In [ ]:
N_FEATURES = len(FEATURE_COLS)  # 10

model = CrimeTCN(
    n_features   = N_FEATURES,
    num_channels = [64, 64, 64, 64],
    kernel_size  = 3,
    dropout      = 0.2,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'CrimeTCN trainable parameters: {n_params:,}')
print()
print(model)

# Smoke test: forward pass with a random batch of 4 sequences.
dummy = torch.randn(4, SEQ_LEN, N_FEATURES).to(DEVICE)
out   = model(dummy)
print(f'\nSmoke test input:  {tuple(dummy.shape)}')
print(f'Smoke test output: {tuple(out.shape)}  (expected (4, 1))')

## Step 8 — Training

In [ ]:
BATCH_SIZE = 32
LR         = 1e-3
EPOCHS     = 50

# MSELoss: mean((predicted - actual)^2). Squared errors penalise large mistakes
# more heavily than small ones — useful because we want to avoid predicting
# wildly wrong counts in high-crime months.
criterion = nn.MSELoss()

# Adam: tracks a running mean (m) and variance (v) of the gradient for every
# parameter and adapts the effective learning rate per parameter.
# It almost always works better than plain SGD for time-series models.
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# shuffle=True for training: breaks temporal order within the batch so the
# model doesn't memorise the order of community areas in each epoch.
# shuffle=False for val/test: metrics must be computed in a reproducible order.
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches:   {len(val_loader)}')
print(f'Test batches:  {len(test_loader)}')

In [ ]:
def run_epoch(loader, train: bool = False) -> float:
    """
    One full pass through the loader. Returns average MSE loss.

    model.train(True)  — enables dropout (adds noise to prevent overfitting)
    model.train(False) — disables dropout (deterministic evaluation)
    """
    model.train(train)
    total_loss, total_rows = 0.0, 0

    for x_batch, y_batch in loader:
        x_batch = x_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        if train:
            # PyTorch accumulates gradients across backward() calls by default.
            # This is useful for gradient accumulation tricks but harmful here.
            # Always zero_grad() before each forward-backward cycle.
            optimizer.zero_grad()

        preds = model(x_batch)           # Forward pass: run the TCN
        loss  = criterion(preds, y_batch) # Compute MSE

        if train:
            loss.backward()              # Compute d(loss)/d(param) for all params
            # Gradient clipping: scale down the gradient vector if its L2 norm
            # exceeds max_norm. Prevents a single large batch from making a
            # destructive weight update (especially early in training).
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()             # param -= lr * grad

        # Weight the batch loss by batch size for a correct epoch average.
        total_loss += loss.item() * len(x_batch)
        total_rows += len(x_batch)

    return total_loss / max(total_rows, 1)


history = {'train_loss': [], 'val_loss': []}

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(train_loader, train=True)

    # torch.no_grad(): disables the gradient tape entirely during evaluation.
    # This halves memory usage and speeds up inference — no gradients needed
    # because we are not calling backward().
    with torch.no_grad():
        val_loss = run_epoch(val_loader, train=False)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)

    if epoch == 1 or epoch % 10 == 0:
        print(f'Epoch {epoch:02d}  train MSE: {train_loss:,.1f}  val MSE: {val_loss:,.1f}')

print('\nTraining complete.')

## Step 9 — Evaluation

In [ ]:
def collect_predictions(loader):
    model.eval()
    preds, actuals = [], []
    with torch.no_grad():
        for x_batch, y_batch in loader:
            batch_preds = model(x_batch.to(DEVICE)).cpu().numpy().ravel()
            preds.extend(batch_preds.tolist())
            actuals.extend(y_batch.numpy().ravel().tolist())
    # Clamp at 0: crime counts can't be negative. Linear layers can output negatives.
    return np.array(actuals), np.maximum(np.array(preds), 0.0)


test_actuals, test_preds = collect_predictions(test_loader)
tcn_mae  = mean_absolute_error(test_actuals, test_preds)
tcn_rmse = rmse_score(test_actuals, test_preds)

print('── Test Results ─────────────────────────────────────')
print(f'Naive baseline  MAE: {naive_mae:.2f}   RMSE: {naive_rmse:.2f}')
print(f'TCN             MAE: {tcn_mae:.2f}   RMSE: {tcn_rmse:.2f}')
print(f'TCN beats naive: {tcn_mae < naive_mae}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: training curve.
# A good training run shows both curves decreasing and converging.
# If val_loss rises while train_loss falls: overfitting — add dropout or reduce capacity.
# If both plateau early: underfitting — increase epochs, add capacity, or lower lr.
axes[0].plot(history['train_loss'], label='Train MSE')
axes[0].plot(history['val_loss'],   label='Val MSE')
axes[0].set_title('Training Curve')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()

# Right: actual vs predicted scatter.
# Points near the red diagonal = accurate predictions.
# Systematic bias (cloud above/below line) = model consistently over/under-predicts.
axes[1].scatter(test_actuals, test_preds, alpha=0.35, s=12)
axes[1].plot([0, test_actuals.max()], [0, test_actuals.max()], 'r--', lw=1.5,
             label='Perfect prediction')
axes[1].set_title('Actual vs Predicted (test 2024)')
axes[1].set_xlabel('Actual count')
axes[1].set_ylabel('Predicted count')
axes[1].legend()

plt.tight_layout()
plt.show()

## Step 10 — `predict()` — App-Facing Contract

This function is the interface between the trained model and `queryMonthlyCrime.ts`.
It accepts a community area number and a reference month, then returns the three
fields the TypeScript layer expects.

In [ ]:
# Build per-CA validation error for confidence scoring.
val_actuals_arr, val_preds_arr = collect_predictions(val_loader)
val_meta = val_dataset.metadata().reset_index(drop=True)
val_meta['actual']    = val_actuals_arr
val_meta['predicted'] = val_preds_arr
val_meta['abs_error'] = (val_meta['actual'] - val_meta['predicted']).abs()

ca_avg_val_error = val_meta.groupby('community_area')['abs_error'].mean().to_dict()
max_val_error    = max(ca_avg_val_error.values()) if ca_avg_val_error else 1.0

print(f'Val rows used for confidence: {len(val_meta)}')
print(f'Max CA avg val error: {max_val_error:.2f}')

In [ ]:
def predict(community_area: int, year: int, month: int) -> dict:
    """
    Return next-month crime prediction for a given Chicago community area.

    Args:
        community_area: CA number 1-77
        year:  the last observed month's year  (e.g. 2024)
        month: the last observed month's month (e.g. 6 = June)

    Returns:
        {
          'predicted_count':  float,
          'trend_direction':  'up' | 'down' | 'stable',
          'confidence_score': float in [0, 1],
        }

    HOW IT WORKS:
      1. Pull the most recent SEQ_LEN feature rows for this CA
         ending at the given (year, month).
      2. Scale them with the training scaler (same stats as training).
      3. Add a batch dimension (the model expects a batch, not a single window).
      4. Run one TCN forward pass.
      5. Compare to lag_12 (same month last year) for trend direction.
      6. Map this CA's validation MAE to a 0-1 confidence score.
    """
    ca_history = feature_df_unscaled[
        (feature_df_unscaled['community_area'] == community_area)
        & (
            (feature_df_unscaled['Year'] < year)
            | ((feature_df_unscaled['Year'] == year) & (feature_df_unscaled['Month'] <= month))
        )
    ].sort_values(['Year', 'Month']).tail(SEQ_LEN)

    if len(ca_history) < SEQ_LEN:
        raise ValueError(
            f'Need {SEQ_LEN} rows before {year}-{month:02d} for CA {community_area}. '
            f'Got {len(ca_history)}.'
        )

    scaled = scaler.transform(ca_history[FEATURE_COLS]).astype(np.float32)

    # unsqueeze(0) inserts a batch dimension at position 0:
    #   (seq_len, features) -> (1, seq_len, features)
    # The model always expects a batch dimension, even for a single example.
    x = torch.tensor(scaled, dtype=torch.float32).unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.no_grad():
        predicted_count = float(model(x).cpu().item())
    predicted_count = max(predicted_count, 0.0)  # Counts can't be negative

    lag12_count = float(ca_history.iloc[-1]['lag_12'])
    if lag12_count <= 0:
        trend_direction = 'stable'
    else:
        ratio = predicted_count / lag12_count
        trend_direction = 'up' if ratio > 1.05 else ('down' if ratio < 0.95 else 'stable')

    ca_error         = ca_avg_val_error.get(community_area, max_val_error)
    confidence_score = float(np.clip(1.0 - ca_error / max(max_val_error, 1e-6), 0.0, 1.0))

    return {
        'predicted_count':  round(predicted_count, 1),
        'trend_direction':  trend_direction,
        'confidence_score': round(confidence_score, 3),
    }

In [ ]:
# Demo: Hyde Park (CA 41), January through June 2024.
print('Hyde Park (CA 41) predictions — Jan through Jun 2024\n')
for m in range(1, 7):
    try:
        result = predict(41, 2024, m)
        print(f'  2024-{m:02d}: {result}')
    except ValueError as e:
        print(f'  2024-{m:02d}: {e}')

## Gotchas Checklist

| # | Gotcha | Why it matters |
|---|--------|----------------|
| 1 | Sort by CA → Year → Month before `shift()` | Prevents lag bleed across community areas |
| 2 | Fit `StandardScaler` on train rows only | Scaling with val/test stats leaks future distribution |
| 3 | `transpose(1, 2)` before Conv1d | Conv1d is channels-first; DataLoader is time-first |
| 4 | `Chomp1d` removes right-side padding | Without it, Conv1d peeks at future timesteps |
| 5 | Weight norm, not Batch norm | BatchNorm is unstable at batch_size=1 during `predict()` |
| 6 | `clip_grad_norm_(max_norm=1.0)` | Prevents huge steps when gradients spike early in training |
| 7 | `max(predicted_count, 0)` | Linear output can be negative; crime counts must be ≥ 0 |
| 8 | `unsqueeze(0)` in `predict()` | Model needs a batch dimension even for a single window |
| 9 | `make_dataset_with_context` for val/test | Val/test windows need prior-split rows as lookback context |
| 10 | `target = shift(-1)` | Predicts NEXT month — using `shift(1)` would predict the previous month |
| 11 | `torch.no_grad()` at inference | Skips gradient tracking → half the memory, faster forward pass |